In [ ]:
#Import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, metrics
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern
from scipy.stats import qmc


In [ ]:
X = np.array([[0.31940389, 0.76295937],
              [0.57432921, 0.8798981],
              [0.73102363, 0.73299988],
              [0.84035342, 0.26473161],
              [0.65011406, 0.68152635],
              [0.41043714, 0.1475543],
              [0.31269116, 0.07872278],
              [0.68341817, 0.86105746],
              [0.08250725, 0.40348751],
              [0.88388983, 0.58225397],
              [0.650114, 0.681526],
              [0.073440, 0.995966],
              [0.834771, 0.764338],
              [0.705169, 0.346253],
              [0.569943, 0.115525],
              [0.388117, 0.661867],
              [0.777293, 0.530146],
              [0.461856, 0.827700],
              [0.507675, 0.360105],
              [0.258010, 0.434197],
              [0.213774, 0.477064],
              [0.167806, 0.824660]

])
y = np.array([1.32267704E-79, 1.03307824e-046, 7.71087511e-016, 3.34177101e-124,
             -3.60606264e-003, -2.15924904e-054, -2.08909327e-091, 2.53500115e-040,
             3.60677119e-081, 6.22985647E-48, -0.00360622, 1.27E-308,
             -2.18E-43, -2.53E-60, 3.9132889736602613e-81, -8.028842574710586e-42,
              -1.0449004136393922e-22, -2.820014336481188e-48, 1.2205190906926124e-8,
              1.7777916949132221e-19, 2.016991536589848e-33, 1.4537597105964157e-158
              ])

In [ ]:
#shape of X & y
print(X.shape)
print(y.shape)

(21, 2)
(21,)


In [ ]:


#GP setup
kernel = ConstantKernel(1.0) * Matern(
    nu=2.5,
    length_scale=[1.0, 1.0],
    length_scale_bounds=(1e-3, 1e4)
)

gpr = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=20,
    alpha=1e-6,
    normalize_y=True
)

# Fit on your 2D data
gpr.fit(X, y)

# Check what the GP learned
print("GP Model Diagnostics:")
print(f"  Kernel: {gpr.kernel_}")
print(f"  Length scales: {gpr.kernel_.k2.length_scale}")
print(f"  Training score: {gpr.score(X, y):.3f}") # Reshape y for scoring

# Define bounds for 2D space
bounds = [
    (X[:, 0].min(), X[:, 0].max()),  # x bounds
    (X[:, 1].min(), X[:, 1].max())   # x bounds
]
print(f"  Bounds: {bounds}")

# Generate 2D candidates using Latin Hypercube
sampler = qmc.LatinHypercube(d=2)
X_candidates = qmc.scale(
    sampler.random(n=5000),
    l_bounds=[bounds[0][0], bounds[1][0]],
    u_bounds=[bounds[0][1], bounds[1][1]]
)

# Predict on candidates
y_pred, y_std = gpr.predict(X_candidates, return_std=True)

# UCB acquisition function
kappa = 2.0
ucb = y_pred + kappa * y_std

# Find best point
best_idx = np.argmax(ucb)
x_next = X_candidates[best_idx]

print(f"\nNext Point to Sample:")
print(f"  X = {x_next}")
print(f"  Predicted y = {y_pred[best_idx]:.4f}")
print(f"  Uncertainty = {y_std[best_idx]:.4f}")
print(f"  UCB score = {ucb[best_idx]:.4f}")

# Show top 5 candidates
top5_idx = np.argsort(ucb)[-5:][::-1]
print(f"\nTop 5 Candidates:")
for i, idx in enumerate(top5_idx, 1):
    print(f"  {i}. X={X_candidates[idx]}, "
          f"pred={y_pred[idx]:.3f}, std={y_std[idx]:.3f}, ucb={ucb[idx]:.3f}")
    # add X candidate to X
    X = np.vstack((X, X_candidates[idx]))
    # Reshape y1 to (N, 1) if it's not already, and reshape the new prediction to (1, 1)
    # It's better to ensure y is always (N, 1) from the start, but for an immediate fix:
    if y.ndim == 1:
        y = y.reshape(-1, 1)
    y = np.vstack((y, y_pred[idx]))

GP Model Diagnostics:
  Kernel: 0.798**2 * Matern(length_scale=[0.024, 1e+04], nu=2.5)
  Length scales: [2.40208302e-02 1.00000000e+04]
  Training score: 1.000
  Bounds: [(np.float64(0.07344), np.float64(0.88388983)), (np.float64(0.07872278), np.float64(0.995966))]

Next Point to Sample:
  X = [0.128075  0.9367313]
  Predicted y = -0.0002
  Uncertainty = 0.0008
  UCB score = 0.0014

Top 5 Candidates:
  1. X=[0.128075  0.9367313], pred=-0.000, std=0.001, ucb=0.001
  2. X=[0.12835605 0.16825388], pred=-0.000, std=0.001, ucb=0.001
  3. X=[0.12802283 0.9187543 ], pred=-0.000, std=0.001, ucb=0.001
  4. X=[0.12841445 0.28156764], pred=-0.000, std=0.001, ucb=0.001
  5. X=[0.12786835 0.63798202], pred=-0.000, std=0.001, ucb=0.001


/usr/local/lib/python3.12/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified upper bound 10000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [ ]:
print (X1)
print (y1)
print("shape of X: ", X1.shape)
print("shape of y: ", y1.shape)

[[0.31940389 0.76295937]
 [0.57432921 0.8798981 ]
 [0.73102363 0.73299988]
 [0.84035342 0.26473161]
 [0.65011406 0.68152635]
 [0.41043714 0.1475543 ]
 [0.31269116 0.07872278]
 [0.68341817 0.86105746]
 [0.08250725 0.40348751]
 [0.88388983 0.58225397]
 [0.650114   0.681526  ]
 [0.07344    0.995966  ]
 [0.70516934 0.3462539 ]
 [0.70510065 0.40616698]
 [0.70539628 0.91821801]
 [0.70493298 0.28398822]
 [0.70555195 0.91074699]]
[[ 1.32267704e-079]
 [ 1.03307824e-046]
 [ 7.71087511e-016]
 [ 3.34177101e-124]
 [-3.60606264e-003]
 [-2.15924904e-054]
 [-2.08909327e-091]
 [ 2.53500115e-040]
 [ 3.60677119e-081]
 [ 6.22985647e-048]
 [-3.60621980e-003]
 [ 1.25250669e-308]
 [ 1.10598442e-004]
 [ 1.11456592e-004]
 [ 1.07784212e-004]
 [ 1.13563392e-004]
 [ 1.05873169e-004]]
shape of X:  (17, 2)
shape of y:  (17, 1)


In [ ]:
#save to files
np.save('/content/drive/MyDrive/AI Data/f1initial_inputs.npy', X1)
np.save('/content/drive/MyDrive/AI Data/f1initial_outputs.npy', y1)

In [ ]:
X1 = np.load('/content/drive/MyDrive/AI Data/f1initial_inputs.npy')
y1 = np.load('/content/drive/MyDrive/AI Data/f1initial_outputs.npy')
print("shape of X: ", X1.shape)
print("shape of y: ", y1.shape)

shape of X:  (17, 2)
shape of y:  (17, 1)


Trying PyTorch & TensorFlow or this week

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from scipy.stats import qmc

In [ ]:
# STEP 1: Define PyTorch Model
# ========================================
class NNSurrogate(nn.Module):
    def __init__(self, input_dim, hidden_sizes=[64, 32], dropout=0.2):
        super(NNSurrogate, self).__init__()

        layers = []
        prev_size = input_dim

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size

        layers.append(nn.Linear(prev_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [ ]:
#========================================
# STEP 2: Train on Your Data
# ========================================
def train_on_your_data(X, y, epochs=1000, lr=0.001):
    """
    Train PyTorch model on your X and y
    """
    n_samples, input_dim = X.shape
    print(f"Training on {n_samples} samples, {input_dim}D")

    # Convert to tensors
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.FloatTensor(y).reshape(-1, 1)

    # Normalize
    X_mean, X_std = X_tensor.mean(0), X_tensor.std(0) + 1e-8
    y_mean, y_std = y_tensor.mean(), y_tensor.std() + 1e-8

    X_norm = (X_tensor - X_mean) / X_std
    y_norm = (y_tensor - y_mean) / y_std

    # Create model
    model = NNSurrogate(input_dim, hidden_sizes=[64, 32], dropout=0.2)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.01)

    # Train
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_norm)
        loss = criterion(predictions, y_norm)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.6f}")

    # Evaluate on training data
    model.eval()
    with torch.no_grad():
        train_pred = model(X_norm)
        train_pred_denorm = train_pred * y_std + y_mean
        mse = ((train_pred_denorm - y_tensor) ** 2).mean().item()
        r2 = 1 - mse / y_tensor.var().item()

    print(f"\nTraining Results:")
    print(f"  MSE: {mse:.6f}")
    print(f"  R²: {r2:.4f}")

    return model, X_mean, X_std, y_mean, y_std

In [ ]:
# Train the model
model, X_mean, X_std, y_mean, y_std = train_on_your_data(X, y, epochs=1000)

Training on 15 samples, 2D
Epoch 200/1000, Loss: 0.241914
Epoch 400/1000, Loss: 0.118564
Epoch 600/1000, Loss: 0.082644
Epoch 800/1000, Loss: 0.023319
Epoch 1000/1000, Loss: 0.098359

Training Results:
  MSE: 0.000000
  R²: 0.9862


In [ ]:
# ========================================
# STEP 3: Predict on New Points
# ========================================
def predict_new_points(model, X_new, X_mean, X_std, y_mean, y_std):
    """
    Predict on new X points
    """
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_new)
        X_norm = (X_tensor - X_mean) / X_std
        y_pred_norm = model(X_norm)
        y_pred = y_pred_norm * y_std + y_mean

    return y_pred.numpy().flatten()

# generating new candidates

n_dims = X.shape[1]

# Define bounds for 2D space
bounds = [
    (X[:, 0].min(), X[:, 0].max()),  # x1 bounds
    (X[:, 1].min(), X[:, 1].max())   # x2 bounds
]

# Generate candidates
sampler = qmc.LatinHypercube(d=n_dims)
X_candidates = qmc.scale(
    sampler.random(n=10000),
    l_bounds=[b[0] for b in bounds],
    u_bounds=[b[1] for b in bounds]
 )


y_pred = predict_new_points(model, X_candidates, X_mean, X_std, y_mean, y_std)
print(f"\nPredictions on new points:")
print(f"X_new: {X_candidates}")
print(f"y_pred: {y_pred}")

#print first X candidate and y pred value
print(f"First X candidate: {X_candidates[0]}")
print(f"First y prediction: {y_pred[0]}")


Predictions on new points:
X_new: [[0.38811779 0.66186726]
 [0.60147534 0.81515324]
 [0.23233632 0.7050022 ]
 ...
 [0.27255837 0.91836138]
 [0.14763717 0.54093263]
 [0.71913239 0.41800541]]
y_pred: [-0.00025204 -0.0003158  -0.00013273 ... -0.00010689 -0.00010432
 -0.00019324]
First X candidate: [0.38811779 0.66186726]
First y prediction: -0.0002520363195799291
